# Phase 9 — FastAPI backend

Wraps `Retriever` (`src/retrieval.py`) → `get_production_generator()` (`src/generation.py`) behind an HTTP API, in `src/api.py`. Two endpoints:

- `GET /health` — liveness check
- `POST /ask` — `{"question": str, "top_k": int (default 10)}` → `{"answer": str, "citations": [...], "sources": [...]}`

`Retriever` and the generator are expensive to construct (loads all 739 chunks, builds the BM25 index, connects to Pinecone/Cohere/OpenRouter) so they're built once at startup via FastAPI's `lifespan` handler, not per-request.

This notebook starts the actual server as a subprocess and exercises it with real HTTP requests, so the demo is genuinely end-to-end and reproducible — not just a description of the code.

Requires `COHERE_API_KEY`, `PINECONE_API_KEY`, `OPENROUTER_API_KEY` in `.env`. Kernel: `rag311`.

In [1]:
import subprocess
import sys
import time
from pathlib import Path

import requests

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
BASE_URL = "http://127.0.0.1:8010"

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.api:app", "--port", "8010", "--host", "127.0.0.1"],
    cwd=PROJECT,
)

for _ in range(30):
    try:
        if requests.get(f"{BASE_URL}/health", timeout=1).status_code == 200:
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    raise RuntimeError("server did not become healthy in time")

print(requests.get(f"{BASE_URL}/health").json())

{'status': 'ok'}


## English question

In [2]:
resp = requests.post(f"{BASE_URL}/ask", json={"question": "How many paid vacation days do I get per year, and does that go up the longer I stay?"})
d = resp.json()
print("status:", resp.status_code)
print("answer:", d["answer"])
print("citations:", d["citations"])
print("n sources:", len(d["sources"]))

status: 200
answer: According to Labor Law 1996, Article 61, every worker is entitled to an annual leave with full pay for a period of fourteen days for each year of service. If the worker has spent five consecutive years in service with the same employer, the annual leave duration increases to twenty-one days. 

Therefore, yes, the number of paid vacation days increases the longer you stay with the same employer.

This information is supported by Labor Law 1996, Article 61.

Please note that this is general information, not legal advice, and you should consult a licensed lawyer for your specific situation.
citations: [{'article_number': '61', 'in_retrieved_context': True}]
n sources: 10


## Arabic question

In [3]:
resp = requests.post(f"{BASE_URL}/ask", json={"question": "كم يوم إجازة سنوية مدفوعة الأجر أستحق؟"})
d = resp.json()
print("status:", resp.status_code)
print("answer:", d["answer"])
print("citations:", d["citations"])

status: 200
answer: يحق لك الحصول على إجازة سنوية مدفوعة الأجر لمدة أربعة عشر يوماً عن كل سنة خدمة. وإذا كنت قد أمضيت في الخدمة لدى صاحب العمل نفسه خمس سنوات متصلة، فإن مدة الإجازة السنوية تصبح واحداً وعشرين يوماً. (قانون العمل 1996، المادة 61).

يرجى ملاحظة أن هذه المعلومات هي للإشارة العامة فقط وليست نصيحة قانونية، وينبغي عليك استشارة محامٍ مرخص للحصول على نصيحة تتعلق بحالتك الخاصة.
citations: [{'article_number': '61', 'in_retrieved_context': True}]


## Input validation

Pydantic enforces `question` non-empty (max 1000 chars) and `top_k` in [1, 20] -- both rejected with a 422 before any retrieval/generation cost is incurred.

In [4]:
print("empty question:", requests.post(f"{BASE_URL}/ask", json={"question": ""}).status_code)
print("top_k too large:", requests.post(f"{BASE_URL}/ask", json={"question": "test", "top_k": 100}).status_code)

empty question: 422
top_k too large: 422


In [5]:
server.terminate()
server.wait(timeout=10)
print("server stopped")

server stopped


## Result

Backend works end-to-end: English and Arabic questions both retrieve the correct article (Labor Law 1996, Article 61) and generate correctly-cited, correctly-grounded answers through the actual HTTP API, not just the underlying Python objects. Input validation rejects malformed requests before any API cost is incurred.

Dependencies pinned in `requirements.txt` (added `fastapi`, `uvicorn` this phase). Phase 9 done -- see `ROADMAP.md`. Next: Phase 10 (frontend) and Phase 11 (deploy this service to a free/cheap host -- Render/Railway/Fly.io -- and wire CORS to the actual frontend origin instead of `"*"`).